
# Laboratorium — Algorytmy AI w grach
## Minimax i alpha-beta pruning

### Cele laboratorium
Po wykonaniu laboratorium powinieneś/powinnaś umieć:

- reprezentować grę jako drzewo stanów,
- zaimplementować algorytm `minimax`,
- zaimplementować `alpha-beta pruning`,
- porównać liczbę odwiedzonych stanów,
- zbudować prostego agenta do gry `tic-tac-toe`.

---

## Zakres
W laboratorium pracujemy na grze **tic-tac-toe**.  
To dobra gra dydaktyczna, ponieważ:

- ma prostą reprezentację stanu,
- pozwala prześledzić działanie drzewa gry,
- da się zaimplementować pełne przeszukiwanie.

Na końcu notebooka znajduje się też **zadanie dodatkowe** dotyczące rozbudowy rozwiązania.


In [ ]:

from __future__ import annotations

from dataclasses import dataclass
import matplotlib.pyplot as plt
from math import inf
from typing import Optional



## Reprezentacja gry

Planszę reprezentujemy jako listę 9 pól:

- `"X"` – ruch gracza X
- `"O"` – ruch gracza O
- `" "` – puste pole

Indeksy pól:

```text
0 | 1 | 2
---------
3 | 4 | 5
---------
6 | 7 | 8
```


In [ ]:

Board = list[str]


def create_board() -> Board:
    return [" "] * 9

fig_counter = 0

def print_board(
    board: Board,
    highlights: list[int] | None = None,
    title: str | None = None,
    transparent_background: bool = False,
) -> None:
    highlights = set(highlights or [])

    fig, ax = plt.subplots(figsize=(4, 4))
    if transparent_background:
        fig.patch.set_alpha(0)
        ax.set_facecolor("none")

    for row in range(3):
        for col in range(3):
            field = row * 3 + col
            y = 2 - row

            if field in highlights:
                ax.add_patch(
                    plt.Rectangle((col, y), 1, 1, facecolor="#ffd54f", alpha=0.45, edgecolor="none")
                )

            mark = board[field].strip()
            if mark:
                ax.text(col + 0.5, y + 0.5, mark, ha="center", va="center", fontsize=28)

    for offset in range(4):
        ax.plot([offset, offset], [0, 3], color="black", linewidth=2)
        ax.plot([0, 3], [offset, offset], color="black", linewidth=2)

    ax.set_xlim(0, 3)
    ax.set_ylim(0, 3)
    ax.set_aspect("equal")
    ax.axis("off")

    if title is not None:
        ax.set_title(title)

    global fig_counter
    # plt.savefig(f"board_{fig_counter}.png", transparent=transparent_background)
    fig_counter += 1
    plt.show()


In [ ]:

board = create_board()
print_board(board)



## Funkcje pomocnicze

Potrzebujemy funkcji, które:

- zwracają listę dostępnych ruchów,
- sprawdzają zwycięzcę,
- rozpoznają stan końcowy,
- zwracają kolejnego gracza.


In [ ]:

WIN_LINES = [
    (0, 1, 2),
    (3, 4, 5),
    (6, 7, 8),
    (0, 3, 6),
    (1, 4, 7),
    (2, 5, 8),
    (0, 4, 8),
    (2, 4, 6),
]


def available_moves(board: Board) -> list[int]:
    return [i for i, cell in enumerate(board) if cell == " "]


def winner(board: Board) -> Optional[str]:
    for a, b, c in WIN_LINES:
        if board[a] != " " and board[a] == board[b] == board[c]:
            return board[a]
    return None


def is_final(board: Board) -> bool:
    return winner(board) is not None or all(cell != " " for cell in board)


def next_player(board: Board) -> str:
    x_count = board.count("X")
    o_count = board.count("O")
    return "X" if x_count == o_count else "O"


def make_move(board: Board, move: int, player: str) -> Board:
    new_board = board.copy()
    new_board[move] = player
    return new_board


In [ ]:

sample_board = ["X", "O", "X",
                " ", "O", " ",
                " ", "X", " "]

print_board(sample_board, title="Przykładowa plansza", highlights=[1, 4, 7])
print("Dostępne ruchy:", available_moves(sample_board))
print("Zwycięzca:", winner(sample_board))
print("Czy stan końcowy:", is_final(sample_board))
print("Kolejny gracz:", next_player(sample_board))



## Funkcja użyteczności

Dla stanów końcowych przyjmujemy:

- wygrana `X` → `+1`
- wygrana `O` → `-1`
- remis → `0`

Zakładamy, że:

- `X` jest graczem **MAX**
- `O` jest graczem **MIN**


In [ ]:

def utility(board: Board) -> int:
    w = winner(board)
    if w == "X":
        return 1
    if w == "O":
        return -1
    return 0



## Minimax

Algorytm:

- dla gracza `X` wybieramy maksimum,
- dla gracza `O` wybieramy minimum.

Ponieważ tic-tac-toe jest małe, możemy przeszukać całe drzewo gry.


In [ ]:

@dataclass
class MinimaxResult:
    score: int
    move: Optional[int]
    visited_nodes: int


In [ ]:
title = "Przykładowa plansza"


def minimax(board: Board, player: str, path: list = []) -> MinimaxResult:
    if is_final(board):
        return MinimaxResult(score=utility(board), move=None, visited_nodes=1)

    visited_nodes = 1
    best_move = None

    if player == "X":
        best_score = -inf
        for move in available_moves(board):
            child = make_move(board, move, player)
            result = minimax(child, "O", path + [str(move)])
            # print_board(child, highlights=[move], title=f"{'-'.join(path + [str(move)])}\n{player} - {result.score}")
            visited_nodes += result.visited_nodes

            if result.score > best_score:
                best_score = result.score
                best_move = move


        return MinimaxResult(score=int(best_score), move=best_move, visited_nodes=visited_nodes)

    best_score = inf
    for move in available_moves(board):
        child = make_move(board, move, player)
        result = minimax(child, "X", path + [str(move)])
        # print_board(child, highlights=[move], title=f"{'-'.join(path + [str(move)])}\n{player} - {result.score}")
        visited_nodes += result.visited_nodes

        if result.score < best_score:
            best_score = result.score
            best_move = move


    return MinimaxResult(score=int(best_score), move=best_move, visited_nodes=visited_nodes)


## Test — minimax

In [ ]:
board = [
    "X", "O", "X",
    " ", "O", " ",
    " ", " ", "X",
]

player = next_player(board)
print_board(board, title=f"Stan początkowy\nRuch gracza: {player}")
print("Ruch gracza:", player)

result = minimax(board, player)
print("\nWynik minimax:")
print("Najlepszy ruch:", result.move)
print("Ocena:", result.score)
print("Odwiedzone węzły:", result.visited_nodes)



## Alpha-beta pruning

Optymalizacja minimax:

- `alpha` – najlepsza znana wartość dla MAX,
- `beta` – najlepsza znana wartość dla MIN.

In [ ]:

def minimax_alpha_beta(
    board: Board,
    player: str,
    alpha: float = -inf,
    beta: float = inf,
    path: list = [],
) -> MinimaxResult:
    # TODO - implementacja minimax z alfa-beta
    return MinimaxResult(score=0, move=None, visited_nodes=1)


## Test — alpha-beta

In [ ]:
board = [
    "X", "O", "X",
    " ", "O", " ",
    " ", " ", "X",
]

player = next_player(board)
print_board(board, title=f"Stan początkowy\nRuch gracza: {player}")

plain = minimax(board, player)
ab = minimax_alpha_beta(board, player)

print("\nMinimax:")
print("Ruch:", plain.move, "Ocena:", plain.score, "Węzły:", plain.visited_nodes)

print("\nAlpha-beta:")
print("Ruch:", ab.move, "Ocena:", ab.score, "Węzły:", ab.visited_nodes)



## Agent do gry

Teraz zbudujemy prostego agenta:

- agent `X` używa minimax albo alpha-beta,
- gracz `O` może wykonywać ruchy ręcznie.

Dla wygody zrobimy funkcję `play_game`.


In [ ]:

def choose_best_move(board: Board, use_alpha_beta: bool = True) -> int:
    player = next_player(board)
    if use_alpha_beta:
        result = minimax_alpha_beta(board, player)
    else:
        result = minimax(board, player)

    if result.move is None:
        raise ValueError("Brak dostępnego ruchu")
    return result.move


In [ ]:

def play_game(use_alpha_beta: bool = True) -> None:
    board = create_board()

    while not is_final(board):
        print()
        print_board(board)
        player = next_player(board)

        if player == "X":
            move = choose_best_move(board, use_alpha_beta=use_alpha_beta)
            print(f"\nAgent X wybiera pole: {move}")
            board = make_move(board, move, "X")
        else:
            move = int(input("\nRuch gracza O (0-8): "))
            if move not in available_moves(board):
                print("Niepoprawny ruch, spróbuj ponownie.")
                continue
            board = make_move(board, move, "O")

    print()
    print_board(board)
    w = winner(board)

    if w is None:
        print("\nRemis.")
    else:
        print(f"\nWygrywa: {w}")



## Uruchomienie gry

Odkomentuj poniższą komórkę, aby zagrać przeciwko agentowi.

Uwaga: ta część wymaga interakcji w notebooku.


In [ ]:

# play_game(use_alpha_beta=True)



## Eksperyment porównawczy

Sprawdźmy kilka stanów planszy i porównajmy:

- najlepszy ruch,
- ocenę,
- liczbę odwiedzonych węzłów.


In [ ]:

test_boards = [
    [
        "X", "O", "X",
        " ", "O", " ",
        " ", " ", "X",
    ],
    [
        "X", "O", " ",
        " ", "X", " ",
        "O", " ", " ",
    ],
    [
        "X", " ", " ",
        " ", "O", " ",
        " ", " ", " ",
    ],
]

for idx, board in enumerate(test_boards, start=1):
    print(f"=== Plansza {idx} ===")
    print_board(board)
    player = next_player(board)

    plain = minimax(board, player)
    ab = minimax_alpha_beta(board, player)

    print("Minimax     -> ruch:", plain.move, "| ocena:", plain.score, "| węzły:", plain.visited_nodes)
    print("Alpha-Beta  -> ruch:", ab.move,    "| ocena:", ab.score,    "| węzły:", ab.visited_nodes)
    print()



## Podsumowanie

W tym laboratorium:

- zaimplementowaliśmy reprezentację gry,
- zbudowaliśmy algorytm `minimax`,
- zoptymalizowaliśmy go za pomocą `alpha-beta pruning`,
- wykorzystaliśmy algorytm do stworzenia agenta do gry.

To jest klasyczny przykład AI opartej na:
- drzewach decyzyjnych,
- optymalizacji przeszukiwania,
- modelu racjonalnego przeciwnika.
